## MODELO 5 CAPAS

In [3]:
import os
from pathlib import Path

DATA_DIR = Path(os.environ.get("DATA_DIR", "./data"))
MODELS_DIR = Path(os.environ.get("MODELS_DIR", "./models"))
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import (
    Dense, Dropout, Conv2D, MaxPool2D, BatchNormalization,
    Flatten, Rescaling
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.layers import GlobalAveragePooling2D

# ============================================================
# PASO 0. RUTAS
# ============================================================
ruta_base = DATA_DIR / "OCT2017" / "OCT_SPLIT"

ruta_train = os.path.join(ruta_base, "train")
ruta_val   = os.path.join(ruta_base, "val")
ruta_test  = os.path.join(ruta_base, "test")

ruta_modelo = MODELS_DIR / "modelo_oct_5bloques_desde_cero.keras"

os.makedirs(os.path.dirname(ruta_modelo), exist_ok=True)

IMG_HEIGHT = 160
IMG_WIDTH = 160
BATCH_SIZE = 32
SEED = 42

# ============================================================
# PASO 1. CARGA DE DATASETS
# ============================================================
train_ds = tf.keras.utils.image_dataset_from_directory(
    ruta_train,
    labels="inferred",
    label_mode="int",
    color_mode="grayscale",
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    ruta_val,
    labels="inferred",
    label_mode="int",
    color_mode="grayscale",
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    ruta_test,
    labels="inferred",
    label_mode="int",
    color_mode="grayscale",
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
num_classes = len(class_names)

print("\nClases detectadas:", class_names)
print("Número de clases:", num_classes)

# ============================================================
# PASO 2. PERFORMANCE DEL PIPELINE
# ============================================================
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

# ============================================================
# PASO 3. CALCULAR CLASS WEIGHTS
# ============================================================
# Tomamos todas las etiquetas de train
y_train = np.concatenate([y.numpy() for _, y in train_ds], axis=0)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weight_dict = {i: class_weights[i] for i in range(num_classes)}
print("\nClass weights:", class_weight_dict)

# ============================================================
# PASO 4. DATA AUGMENTATION
# ============================================================
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomTranslation(height_factor=0.03, width_factor=0.03),
    tf.keras.layers.RandomRotation(0.03),
    tf.keras.layers.RandomZoom(0.05),
    tf.keras.layers.RandomContrast(0.08)
], name="data_augmentation")

# ============================================================
# PASO 5. MODELO CNN DESDE CERO - 5 BLOQUES
# ============================================================
model = Sequential([
    tf.keras.layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 1)),

    data_augmentation,
    Rescaling(1./255),

    # Bloque 1
    Conv2D(32, (3, 3), padding="same", activation="relu"),
    BatchNormalization(),
    MaxPool2D((2, 2)),

    # Bloque 2
    Conv2D(64, (3, 3), padding="same", activation="relu"),
    BatchNormalization(),
    MaxPool2D((2, 2)),

    # Bloque 3
    Conv2D(128, (3, 3), padding="same", activation="relu"),
    BatchNormalization(),
    MaxPool2D((2, 2)),

    # Bloque 4
    Conv2D(256, (3, 3), padding="same", activation="relu"),
    BatchNormalization(),
    MaxPool2D((2, 2)),

    # Bloque 5
    Conv2D(512, (3, 3), padding="same", activation="relu"),
    BatchNormalization(),
    MaxPool2D((2, 2)),

    Dropout(0.40),

    tf.keras.layers.GlobalAveragePooling2D(),

    Dense(128, activation="relu"),
    Dropout(0.50),

    Dense(num_classes, activation="softmax")
])

# ============================================================
# PASO 6. COMPILACIÓN
# ============================================================
model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    metrics=["accuracy"]
)

model.summary()

# ============================================================
# PASO 7. CALLBACKS
# ============================================================
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

checkpoint = ModelCheckpoint(
    filepath=ruta_modelo,
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=False,
    mode="min",
    verbose=1
)

# ============================================================
# PASO 8. ENTRENAMIENTO
# ============================================================
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=[early_stopping, reduce_lr, checkpoint],
    class_weight=class_weight_dict,
    verbose=1
)

# ============================================================
# PASO 8.1 CARGAR EL MEJOR MODELO
# ============================================================
print("\nCargando el mejor modelo guardado...")
model = load_model(ruta_modelo)

# ============================================================
# PASO 9. EVALUACIÓN FINAL EN TEST
# ============================================================
test_loss, test_acc = model.evaluate(test_ds, verbose=1)
print("\nTest loss:", test_loss)
print("Test accuracy:", test_acc)

# Obtener y_true
y_true = np.concatenate([y.numpy() for _, y in test_ds], axis=0)

# Obtener probabilidades y predicciones
y_pred_prob = model.predict(test_ds, verbose=1)
y_pred = np.argmax(y_pred_prob, axis=1)

acc = accuracy_score(y_true, y_pred)
print("\nAccuracy score (sklearn):", acc)

print("\nMatriz de confusión:")
print(confusion_matrix(y_true, y_pred))

print("\nReporte de clasificación:")
print(classification_report(y_true, y_pred, target_names=class_names))

# ============================================================
# PASO 10. MODELO FINAL
# ============================================================
print("\nEl mejor modelo quedó guardado en:", ruta_modelo)

Found 58507 files belonging to 4 classes.
Found 12536 files belonging to 4 classes.
Found 12542 files belonging to 4 classes.

Clases detectadas: ['CNV', 'DME', 'DRUSEN', 'NORMAL']
Número de clases: 4

Class weights: {0: np.float64(0.5615521941106462), 1: np.float64(1.8304029533224877), 2: np.float64(2.425663349917081), 3: np.float64(0.7932507185856066)}


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ data_augmentation (Sequential)       │ (None, 160, 160, 1)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ rescaling_2 (Rescaling)              │ (None, 160, 160, 1)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_10 (Conv2D)                   │ (None, 160, 160, 32)        │             320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_10               │ (None, 160, 160, 32)        │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_10 (MaxPooling2D)      │ (None, 80, 80, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_11 (Conv2D)                   │ (None, 80, 80, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_11               │ (None, 80, 80, 64)          │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_11 (MaxPooling2D)      │ (None, 40, 40, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_12 (Conv2D)                   │ (None, 40, 40, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_12               │ (None, 40, 40, 128)         │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_12 (MaxPooling2D)      │ (None, 20, 20, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_13 (Conv2D)                   │ (None, 20, 20, 256)         │         295,168 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_13               │ (None, 20, 20, 256)         │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_13 (MaxPooling2D)      │ (None, 10, 10, 256)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_14 (Conv2D)                   │ (None, 10, 10, 512)         │       1,180,160 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_14               │ (None, 10, 10, 512)         │           2,048 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_14 (MaxPooling2D)      │ (None, 5, 5, 512)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_4 (Dropout)                  │ (None, 5, 5, 512)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d_1           │ (None, 512)                 │              

 Total params: 1,638,148 (6.25 MB)

 Trainable params: 1,636,164 (6.24 MB)

 Non-trainable params: 1,984 (7.75 KB)

Epoch 1/30
1829/1829 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6093 - loss: 1.0183
Epoch 1: val_loss improved from None to 0.74861, saving model to C:\Users\Oscar Ferreira\OneDrive - AUTO LINEAS AMERICA SA DE CV\Escritorio\MCD\5 - APRENDIZAJE PROFUNDO\PROYECTO FINAL\OCT2017\modelo_oct_5bloques_desde_cero.keras

Epoch 1: finished saving model to C:\Users\Oscar Ferreira\OneDrive - AUTO LINEAS AMERICA SA DE CV\Escritorio\MCD\5 - APRENDIZAJE PROFUNDO\PROYECTO FINAL\OCT2017\modelo_oct_5bloques_desde_cero.keras
1829/1829 ━━━━━━━━━━━━━━━━━━━━ 1990s 1s/step - accuracy: 0.7260 - loss: 0.7557 - val_accuracy: 0.7351 - val_loss: 0.7486 - learning_rate: 1.0000e-04
Epoch 2/30
1829/1829 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8425 - loss: 0.4594
Epoch 2: val_loss improved from 0.74861 to 0.48668, saving model to C:\Users\Oscar Ferreira\OneDrive - AUTO LINEAS AMERICA SA DE CV\Escritorio\MCD\5 - APRENDIZAJE PROFUNDO\PROYECTO FINAL\OCT2017\modelo_oct_5bloques_desde_cero.keras

Epoch 2: finishe